In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/';

In [0]:
%sql
SELECT count(*) AS total_rows
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/',
  format => 'csv',
  header => true,
  inferSchema => true,
  pathGlobFilter => 'SST*/**/*.csv',
  recursiveFileLookup => true
);

In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/SST - 1993/';

In [0]:
%sql
SELECT count(*) AS total_rows
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/',
  format => 'csv',
  header => true,
  inferSchema => true,
  recursiveFileLookup => true,
  pathGlobFilter => 'SST*/**/*.csv'
);

In [0]:
%sql
SELECT count(*) AS rows_for_1993_test
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/SST - 1993/SST-1993/',
  format => 'csv',
  header => true,
  inferSchema => true
);
-- Returns 89,424 rows if the file is readable
-- Zero means the path is wrong — check the folder structure


In [0]:
%sql
SELECT count(*) AS total_sst_rows_all_years
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/SST*/',
  format => 'csv',
  header => true,
  inferSchema => true,
  recursiveFileLookup => true
);
-- Reads all SST folders regardless of naming variation
-- Expected: approximately 2.9 million rows across 33 years
-- If zero the SST*/ path pattern is not matching — check folder names

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/SST - 1993/SST-1993/',
  format => 'csv',
  header => true,
  inferSchema => true
)
LIMIT 5;
-- Shows column names and sample data before creating the table
-- We need to confirm column names for Silver queries later
-- Look for time, latitude, longitude and sst columns

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/SST - 1993/SST-1993/',
  format => 'csv',
  header => true,
  inferSchema => true
)
LIMIT 2;
-- Checking if time column exists — it should show date values
-- If missing we need to extract year and month from the file name instead

In [0]:
%sql
SELECT
  _metadata.file_name AS source_file,
  Latitude,
  ` Longitude` AS longitude,
  ` Data Value` AS sst
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/SST - 1993/SST-1993/',
  format => 'csv',
  header => true,
  inferSchema => true
)
LIMIT 5;
-- Column names have leading spaces from the CSV headers
-- Backticks handle the spaces and the space in Data Value
-- source_file will show the filename we use to extract date


In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_sst_master
USING DELTA AS
SELECT
  _metadata.file_name AS source_file,
  Latitude AS latitude,
  ` Longitude` AS longitude,
  ` Data Value` AS sst_raw,
  regexp_extract(_metadata.file_name, '(\\d{4})', 0) AS year,
  regexp_extract(_metadata.file_name, 'SST-([A-Za-z]+)\\d{4}', 1) AS month_name
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/SST*/',
  format => 'csv',
  header => true,
  inferSchema => true,
  recursiveFileLookup => true
);
-- Creates Bronze SST table with year and month extracted from filename
-- regexp_extract pulls the 4-digit year from the filename
-- month_name gives us Jan, Feb, Mar etc — converted to number in Silver
-- All 33 years ingested in one query

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_sst_master
USING DELTA AS
SELECT
  Latitude AS latitude,
  ` Longitude` AS longitude,
  ` Data Value` AS sst_raw
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/SST*/',
  format => 'csv',
  header => true,
  inferSchema => true,
  recursiveFileLookup => true
);
-- Creates Bronze SST table with the three available columns
-- No time column in source files — date extracted from filename in Silver
-- Using SST*/ path to capture all year folders regardless of naming

In [0]:
%sql
SELECT count(*) AS total_rows
FROM mcphersonsharyn_bronze.bronze_sst_master;
-- Should return approximately 3,085,128 rows
-- If zero the path pattern is not matching

In [0]:
%sql
SELECT
  'bronze_sst_master' AS table_name,
  count(*) AS total_rows,
  min(latitude) AS min_lat,
  max(latitude) AS max_lat,
  min(longitude) AS min_lon,
  max(longitude) AS max_lon,
  min(sst_raw) AS min_sst,
  max(sst_raw) AS max_sst
FROM mcphersonsharyn_bronze.bronze_sst_master;
-- Confirms row count and coordinate bounds are within ATLAS bounding box
-- SST values should be between -2 and 40 degrees Celsius
-- Latitude should be between -23 and 22
-- Longitude should be between -73 and 126

In [0]:
%sql
-- NOTE: Longitude range covers full globe (-179 to 180)
-- ATLAS bounding box filter (-73 to 126) applied in Silver
-- NOTE: min_sst of -999 is NOAA missing data sentinel value
-- Removed in Silver with WHERE sst_raw != -999

In [0]:
%sql
SELECT count(*) AS total_cc_rows
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/',
  format => 'csv',
  header => true,
  inferSchema => true,
  pathGlobFilter => '[0-9][0-9][0-9][0-9]/**/*.csv',
  recursiveFileLookup => true
);
-- Counts all CC files across year folders 2002 to 2026
-- Plain year number folders only — 2002, 2003 ... 2026
-- Expected: approximately 600,000 rows

In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/2009/';
-- Shows what files are inside the 2009 CC folder
-- Confirms the folder structure and file names

In [0]:
%sql
SELECT count(*) AS total_cc_rows
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/2009/CC-Jan2009.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);
-- Testing single file read directly
-- If this returns rows we know files are readable
-- Then we just need to fix the path pattern

In [0]:
%sql
SELECT count(*) AS total_2009_rows
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/2009/',
  format => 'csv',
  header => true,
  inferSchema => true,
  pathGlobFilter => 'CC-*.csv'
);
-- Reads all CC monthly files from 2009 folder only
-- pathGlobFilter excludes FullYear file
-- Expected: approximately 89,424 rows (12 months x 7,452 cells)

In [0]:
%sql
SELECT count(*) AS total_cc_all_years
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/20*/',
  format => 'csv',
  header => true,
  inferSchema => true,
  pathGlobFilter => 'CC-*.csv'
);
-- Uses 20*/ to match all year folders starting with 20
-- Covers 2002 through 2026
-- pathGlobFilter CC-*.csv excludes FullYear files
-- Expected: approximately 2.5 million rows across 25 years

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/2009/',
  format => 'csv',
  header => true,
  inferSchema => true,
  pathGlobFilter => 'CC-*.csv'
)
LIMIT 5;
-- Shows column names and sample data
-- Look for latitude, longitude and chlorophyll columns
-- Note exact column names for Silver queries

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_cc_master
USING DELTA AS
SELECT
  Latitude AS latitude,
  ` Longitude` AS longitude,
  ` Data Value` AS cc_raw
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/20*/',
  format => 'csv',
  header => true,
  inferSchema => true,
  pathGlobFilter => 'CC-*.csv'
);
-- Creates Bronze CC master table
-- Same column structure as SST — leading spaces handled with backticks
-- FullYear files excluded by CC-*.csv pattern
-- Date extracted from filename in Silver same as SST
-- 20*/ pattern captures all year folders 2002 to 2026

In [0]:
%sql
SELECT
  'bronze_cc_master' AS table_name,
  count(*) AS total_rows,
  min(latitude) AS min_lat,
  max(latitude) AS max_lat,
  min(cc_raw) AS min_cc,
  max(cc_raw) AS max_cc
FROM mcphersonsharyn_bronze.bronze_cc_master;
-- Confirms row count and value ranges
-- CC values should be positive — negative values are missing data codes
-- Latitude should be between -23 and 22

In [0]:
%sql
SELECT count(*) AS total_cmems_rows
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/',
  format => 'csv',
  header => true,
  inferSchema => true,
  pathGlobFilter => 'cmems-*/**/*.csv',
  recursiveFileLookup => true
);
-- Counts all CMEMS annual CSV files
-- cmems-*/ pattern matches cmems-1993 through cmems-2026
-- Expected: approximately 500,000 rows across 34 years

In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/cmems-2009/';
-- Shows what is inside one CMEMS year folder
-- Confirms file names and structure


In [0]:
%sql
SELECT count(*) AS total_cmems_rows
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-2009/cmems_2009.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);
-- Testing single CMEMS file read directly
-- Expected: approximately 15,000 rows for one year
-- If this returns rows we know files are readable

In [0]:
%sql
SELECT
  min(latitude) AS min_lat,
  max(latitude) AS max_lat,
  min(longitude) AS min_lon,
  max(longitude) AS max_lon,
  count(*) AS total_rows
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-2009/cmems_2009.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);
-- Checks the coordinate range of the CMEMS file
-- If it covers the full globe we need to filter to bounding box
-- Our bounding box: lat -23 to 22, lon -73 to 126


In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-2009/cmems_2009.csv',
  format => 'csv',
  header => true,
  inferSchema => true
)
LIMIT 3;
-- Shows column names and sample data
-- Expected: depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_cmems_raw
USING DELTA AS
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-*/',
  format => 'csv',
  header => true,
  inferSchema => true,
  recursiveFileLookup => true
);
-- Creates Bronze CMEMS table from all 34 annual CSV files
-- cmems-*/ matches all year folders 1993 to 2026
-- Estimated 329 million rows total 

In [0]:
%sql
SELECT count(*) AS test_count
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-1993/',
  format => 'csv',
  header => true,
  inferSchema => true,
  recursiveFileLookup => true
);
-- Tests reading just 1993 folder directly
-- If this returns rows the single folder path works
-- We then build the full path pattern from there

In [0]:
%sql
SELECT count(*) AS test_two_years
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-1993/',
  format => 'csv',
  header => true,
  inferSchema => true,
  recursiveFileLookup => true
)
UNION ALL
SELECT count(*)
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-1994/',
  format => 'csv',
  header => true,
  inferSchema => true,
  recursiveFileLookup => true
);
-- Tests reading two folders with UNION ALL
-- If this works we build the full table the same way
-- Expected: approximately 19.3 million rows (2 years)

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_cmems_raw
USING DELTA AS
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-1993/',
  format => 'csv',
  header => true,
  inferSchema => true,
  recursiveFileLookup => true
);
-- Creates the Bronze CMEMS table using 1993 as the template
-- Establishes the schema — all subsequent years use INSERT INTO
-- Expected: 9,671,220 rows for 1993

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_cmems_raw
USING DELTA AS
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-1993/cmems_1993.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);
-- Reads the 1993 CMEMS file directly using full file path
-- No wildcard or recursiveFileLookup needed
-- Expected: 9,671,220 rows

In [0]:
%sql
-- Test just the SELECT first without CREATE TABLE
SELECT count(*) AS test
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-1993/cmems_1993.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);
-- Testing if SELECT still works on this file
-- If this returns 9,671,220 the issue is with CREATE TABLE not the file

In [0]:
%sql
-- Step 1 — Create empty table with correct schema
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_cmems_raw
(
  depth DOUBLE,
  latitude DOUBLE,
  longitude DOUBLE,
  time STRING,
  uo DOUBLE,
  vo DOUBLE,
  zos DOUBLE,
  mlotst DOUBLE,
  so DOUBLE,
  thetao DOUBLE
)
USING DELTA;
-- Creates empty Delta table matching CMEMS file structure
-- Time stored as STRING at Bronze — cast to DATE in Silver
-- INSERT INTO will populate year by year

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-1993/cmems_1993.csv',
  format => 'csv',
  header => true,
  inferSchema => true
)
LIMIT 0;
-- Returns no rows but shows column names and inferred data types
-- Look at the column headers to confirm exact names and types

In [0]:
%sql
SELECT
  typeof(depth) AS depth_type,
  typeof(latitude) AS lat_type,
  typeof(longitude) AS lon_type,
  typeof(time) AS time_type,
  typeof(uo) AS uo_type,
  typeof(vo) AS vo_type,
  typeof(zos) AS zos_type,
  typeof(mlotst) AS mlotst_type,
  typeof(so) AS so_type,
  typeof(thetao) AS thetao_type
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-1993/cmems_1993.csv',
  format => 'csv',
  header => true,
  inferSchema => true
)
LIMIT 1;
-- Shows the data type inferred for each column
-- We need these to match our Bronze table schema exactly
-- The mismatch is likely on the time column type

In [0]:
%sql
-- Drop the existing empty table and recreate with correct schema
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_cmems_raw
(
  depth DOUBLE,
  latitude DOUBLE,
  longitude DOUBLE,
  time DATE,
  uo DOUBLE,
  vo DOUBLE,
  zos DOUBLE,
  mlotst DOUBLE,
  so DOUBLE,
  thetao DOUBLE
)
USING DELTA;
-- Recreating with time as DATE not STRING
-- This matches the inferred type from the CSV file exactly
-- All other columns confirmed as DOUBLE

In [0]:
%sql
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-1993/cmems_1993.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);
-- First year INSERT INTO Bronze CMEMS table
-- Schema now matches exactly — should work without mismatch error
-- Expected: 9,671,220 rows

In [0]:
%python
# Read 1993 CMEMS file and write to Delta table
df = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("/Volumes/workspace/raw/environmental_data/cmems-1993/cmems_1993.csv")

# Write to Delta table — overwrite creates fresh table with correct schema
df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_cmems_raw")

print(f"Rows written: {df.count()}")
# Uses Python Spark API instead of SQL
# overwrite mode creates the table with exactly the right schema
# from the actual file — no manual schema definition needed

In [0]:
%sql
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/cmems-1994/cmems_1994.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);
-- Tests INSERT INTO for 1994 now table schema exists
-- If this works we can do all years in SQL
-- Expected: 9,671,220 rows

In [0]:
%sql
-- Insert CMEMS 1995
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-1995/cmems_1995.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 1996
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-1996/cmems_1996.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 1997
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-1997/cmems_1997.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 1998
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-1998/cmems_1998.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 1999
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-1999/cmems_1999.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2000
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2000/cmems_2000.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2001
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2001/cmems_2001.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2002
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2002/cmems_2002.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2003
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2003/cmems_2003.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2004
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2004/cmems_2004.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2005
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2005/cmems_2005.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2006
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2006/cmems_2006.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2007
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2007/cmems_2007.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2008
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2008/cmems_2008.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2009
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2009/cmems_2009.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2010
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2010/cmems_2010.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2011
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2011/cmems_2011.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2012
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2012/cmems_2012.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2013
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2013/cmems_2013.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2014
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2014/cmems_2014.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2015
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2015/cmems_2015.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2016
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2016/cmems_2016.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2017
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2017/cmems_2017.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2018
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2018/cmems_2018.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2019
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2019/cmems_2019.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2020
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2020/cmems_2020.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2021
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2021/cmems_2021.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2022
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2022/cmems_2022.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2023
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2023/cmems_2023.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2024
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2024/cmems_2024.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2025
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2025/cmems_2025.csv', format => 'csv', header => true, inferSchema => true);

-- Insert CMEMS 2026
INSERT INTO mcphersonsharyn_bronze.bronze_cmems_raw (depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao)
SELECT depth, latitude, longitude, time, uo, vo, zos, mlotst, so, thetao
FROM read_files('/Volumes/workspace/raw/environmental_data/cmems-2026/cmems_2026.csv', format => 'csv', header => true, inferSchema => true);
-- Final year — 2026 covers January to April only

In [0]:
%sql
SELECT
  YEAR(time) AS year,
  count(*) AS rows
FROM mcphersonsharyn_bronze.bronze_cmems_raw
GROUP BY YEAR(time)
ORDER BY year;
-- Shows which years are currently in the Bronze CMEMS table
-- Confirms which INSERT statements actually ran
-- Each year should show approximately 9,671,220 rows

In [0]:
%sql
SELECT
  'bronze_cmems_raw' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT YEAR(time)) AS distinct_years,
  min(YEAR(time)) AS earliest_year,
  max(YEAR(time)) AS latest_year
FROM mcphersonsharyn_bronze.bronze_cmems_raw;
-- Confirms 34 distinct years 1993 to 2026
-- 2026 has fewer rows — January to April only
-- 322 million rows total — largest table in ATLAS

In [0]:
%sql
SELECT
  'bronze_enso_raw' AS table_name,
  count(*) AS total_rows,
  min(YEAR) AS earliest_year,
  max(YEAR) AS latest_year
FROM mcphersonsharyn_bronze.bronze_enso_raw;
-- ENSO was ingested in original pipeline — validating it still exists
-- Expected: 48 rows, 1979 to 2026

In [0]:
%sql
SELECT
  'bronze_microplastics_raw' AS table_name,
  count(*) AS total_rows
FROM mcphersonsharyn_bronze.bronze_microplastics_raw;
-- NCEI microplastics ingested in original pipeline
-- Expected: 471 rows covering Atlantic and Caribbean
-- If table does not exist we need to create it

In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/AOMIMicrosplastics/';
-- Confirms AOMI file is in the volume before ingestion

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/AOMIMicrosplastics/AOMI_survey_data.csv',
  format => 'csv',
  header => true,
  inferSchema => true
)
LIMIT 3;
-- Shows column names and sample data before creating the table
-- Look for latitude, longitude, date and particle density columns
-- AOMI has 449 columns — we only need a few key ones

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/AOMIMicrosplastics/AOMI_survey_data.csv',
  format => 'csv',
  header => true,
  inferSchema => true
)
LIMIT 3;
-- Skips the 458 comment header rows before data starts
-- Row 459 contains the actual column headers
-- Row 460 onwards is the survey data

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/AOMIMicrosplastics/AOMI_survey_data.csv',
  format => 'csv',
  header => false,
  inferSchema => false
)
WHERE _c0 NOT LIKE '#%'
LIMIT 10;


In [0]:
%sql
SELECT _c0
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/AOMIMicrosplastics/AOMI_survey_data.csv',
  format => 'csv',
  header => false,
  inferSchema => false
)
WHERE _c0 NOT LIKE '#%'
AND _c0 != '0001:Data ID'
LIMIT 3;
-- Shows just the first column of actual data rows
-- Checking if the full row is in _c0 as one long string
-- This would happen if the delimiter is not comma

In [0]:
%python
# AOMI file has 449 columns and 458 comment header rows
# read_files() SQL cannot handle this width reliably
# Python pandas reads it correctly with skiprows

import pandas as pd
from pyspark.sql import SparkSession

# Read with pandas skipping comment rows
df_pandas = pd.read_csv(
    '/Volumes/workspace/raw/environmental_data/AOMIMicrosplastics/AOMI_survey_data.csv',
    skiprows=458,
    encoding='utf-8-sig',
    low_memory=False
)

print(f"Total rows: {len(df_pandas)}")
print(f"Total columns: {len(df_pandas.columns)}")

# Keep only the five columns ATLAS needs
df_slim = df_pandas[[
    '0001:Data ID',
    '0045:GPS_Lat Int',
    '0047:GPS_Lon Int', 
    '0033:Sampling start Date',
    '0192:Particle density_m3 (d<5mm)'
]].copy()

# Rename columns to clean names
df_slim.columns = ['data_id', 'latitude_raw', 'longitude_raw', 
                   'sample_date_raw', 'particle_density_raw']

print(f"Slim dataframe rows: {len(df_slim)}")
print(df_slim.head(3).to_string())

# Convert to Spark and write to Delta
df_spark = spark.createDataFrame(df_slim.astype(str))
df_spark.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("mcphersonsharyn_bronze.bronze_microplastics_aomi_raw")

print("Bronze AOMI table created successfully")
# Python used because read_files() cannot handle 449 column CSV
# Only the 5 key columns needed for ATLAS are kept at Bronze
# All values stored as strings — cast to correct types in Silver

In [0]:
%sql
SELECT
  'bronze_microplastics_aomi_raw' AS table_name,
  count(*) AS total_rows,
  min(CAST(latitude_raw AS DOUBLE)) AS min_lat,
  max(CAST(latitude_raw AS DOUBLE)) AS max_lat,
  min(CAST(sample_date_raw AS DATE)) AS earliest_date,
  max(CAST(sample_date_raw AS DATE)) AS latest_date
FROM mcphersonsharyn_bronze.bronze_microplastics_aomi_raw;
-- Confirms 1,682 rows loaded correctly
-- Latitude range should be within -23 to 22
-- Date range confirms historical coverage

In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/CoralBleaching/';
-- Confirms coral bleaching file is in the volume
-- Should show the dhw_5km CSV file

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/CoralBleaching/dhw_5km_31c8_080d_71d0.csv',
  format => 'csv',
  header => true,
  inferSchema => true
)
LIMIT 3;
-- Shows column names and sample data
-- Expected: time, latitude, longitude, CRW_DHW, CRW_BAA
-- Confirms file is readable before creating Bronze table

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_coral_bleaching_raw
USING DELTA AS
SELECT
  time,
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  CAST(CRW_DHW AS DOUBLE) AS crw_dhw,
  CAST(CRW_BAA AS INTEGER) AS crw_baa
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/CoralBleaching/dhw_5km_31c8_080d_71d0.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)
WHERE time != 'UTC'
AND latitude != 'degrees_north';
-- Creates Bronze coral bleaching table
-- Excludes the units row where time = UTC
-- inferSchema => false used because units row confuses type detection
-- CRW_DHW = Degree Heating Week — bleaching stress measure
-- CRW_BAA = Bleaching Alert Area — 0 to 5 categorical scale

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_coral_bleaching_raw
USING DELTA AS
SELECT
  time,
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  TRY_CAST(CRW_DHW AS DOUBLE) AS crw_dhw,
  TRY_CAST(CRW_BAA AS INTEGER) AS crw_baa
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/CoralBleaching/dhw_5km_31c8_080d_71d0.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)
WHERE time != 'UTC'
AND latitude != 'degrees_north';
-- TRY_CAST used instead of CAST to handle NaN values gracefully
-- NaN values become NULL in the Bronze table
-- NULL values are expected and handled in Silver


In [0]:
%sql
SELECT count(*) AS total_rows
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/CoralBleaching/dhw_5km_31c8_080d_71d0.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)
WHERE time != 'UTC';
-- Simple count to confirm file is readable
-- Excludes units row only
-- No casting — just counting raw rows

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_coral_bleaching_raw
USING DELTA AS
SELECT
  time,
  latitude,
  longitude,
  CRW_DHW AS crw_dhw,
  CRW_BAA AS crw_baa
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/CoralBleaching/dhw_5km_31c8_080d_71d0.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)
WHERE time != 'UTC'
AND latitude != 'degrees_north';
-- Creates Bronze table storing all values as strings
-- No casting at Bronze layer — keeps data exactly as received
-- Casting to correct types happens in Silver
-- 18 million rows expected — will take a few minutes

In [0]:
%sql
CREATE OR REPLACE TABLE mcphersonsharyn_bronze.bronze_coral_bleaching_raw
USING DELTA AS
SELECT
  time,
  latitude,
  longitude,
  CRW_DHW AS crw_dhw,
  CRW_BAA AS crw_baa
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/CoralBleaching/dhw_5km_31c8_080d_71d0.csv',
  format => 'csv',
  header => true,
  inferSchema => false
);
-- Creates Bronze table without WHERE filter
-- Units row will be included but filtered out in Silver
-- Keeping Bronze as raw as possible
-- 18 million rows expected including units row

In [0]:
%python
# Coral bleaching file too large for SQL CREATE TABLE AS SELECT
# Python Spark handles large files more reliably
# Units row (UTC, degrees_north etc) filtered out

df = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "false") \
  .load("/Volumes/workspace/raw/environmental_data/CoralBleaching/dhw_5km_31c8_080d_71d0.csv")

# Filter out the units row
df = df.filter(df.time != "UTC")

# Write to Delta table
df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_coral_bleaching_raw")

print(f"Rows written: {df.count()}")
# Python used because SQL CREATE TABLE AS SELECT fails on large files
# Units row filtered before writing
# All values stored as strings — cast to correct types in Silver

In [0]:
%sql
SELECT
  'bronze_coral_bleaching_raw' AS table_name,
  count(*) AS total_rows,
  count(DISTINCT YEAR(TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'"))) AS distinct_years,
  min(TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS earliest_date,
  max(TO_DATE(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS latest_date
FROM mcphersonsharyn_bronze.bronze_coral_bleaching_raw;
-- Confirms 18 million rows loaded correctly
-- Annual snapshots from 1985 to 2026
-- time column stored as string — parsed here for validation only
-- Casting to DATE happens properly in Silver

In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/GlobalFishing-2012/';
-- Confirms what files are inside the 2012 folder
-- Checking if zip files or CSV files uploaded

In [0]:
%python
# Spark can read CSVs inside zip files directly
df = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("/Volumes/workspace/raw/environmental_data/GlobalFishing-2012/fleet-daily-csvs-100-v3-2012.zip")

print(f"Rows: {df.count()}")
print(f"Columns: {df.columns}")
# Tests if Spark can read directly from zip

In [0]:
%python
import zipfile

# Unzip all Global Fishing Watch zip files
# After unzipping read_files() can access the CSV files directly
years = range(2012, 2025)
base_path = "/Volumes/workspace/raw/environmental_data/"

for year in years:
    zip_path = f"{base_path}GlobalFishing-{year}/fleet-daily-csvs-100-v3-{year}.zip"
    extract_path = f"{base_path}GlobalFishing-{year}/"
    
    try:
        print(f"Unzipping {year}...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_path)
        print(f"Year {year} complete")
    except Exception as e:
        print(f"Year {year} failed: {e}")

print("All years unzipped")


In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/GlobalFishing-2012/';

-- Shows what files are now in the 2012 folder after unzipping
-- Should see CSV files alongside the original zip

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/GlobalFishing-2012/fleet-daily-csvs-100-v3-2012-01-01.csv',
  format => 'csv',
  header => true,
  inferSchema => true
)
LIMIT 3;

-- Shows column names and sample data for one daily file
-- Need to confirm exact column names before creating Bronze table
-- Look for date, lat, lon and fishing_hours columns

In [0]:
%python
# Global Fishing Watch — create Bronze table using Python(finding it difficult to use SQL)
# SQL CREATE TABLE AS SELECT fails on this volume of files
# Python Spark handles multiple daily CSV files reliably

df = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("/Volumes/workspace/raw/environmental_data/GlobalFishing-*/fleet-daily-csvs-100-v3-*.csv")

df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_fishing_watch_raw")

print(f"Rows written: {df.count()}")

# Reads all daily CSV files across all 13 year folders
# GlobalFishing-*/ wildcard matches all year folders
# fleet-daily-csvs-100-v3-*.csv matches all daily files


In [0]:

%sql
SELECT 'bronze_sst_master' AS table_name, count(*) AS rows FROM mcphersonsharyn_bronze.bronze_sst_master
UNION ALL
SELECT 'bronze_cc_master', count(*) FROM mcphersonsharyn_bronze.bronze_cc_master
UNION ALL
SELECT 'bronze_cmems_raw', count(*) FROM mcphersonsharyn_bronze.bronze_cmems_raw
UNION ALL
SELECT 'bronze_enso_raw', count(*) FROM mcphersonsharyn_bronze.bronze_enso_raw
UNION ALL
SELECT 'bronze_microplastics_raw', count(*) FROM mcphersonsharyn_bronze.bronze_microplastics_raw
UNION ALL
SELECT 'bronze_microplastics_aomi_raw', count(*) FROM mcphersonsharyn_bronze.bronze_microplastics_aomi_raw
UNION ALL
SELECT 'bronze_coral_bleaching_raw', count(*) FROM mcphersonsharyn_bronze.bronze_coral_bleaching_raw
UNION ALL
SELECT 'bronze_fishing_watch_raw', count(*) FROM mcphersonsharyn_bronze.bronze_fishing_watch_raw
ORDER BY table_name;

-- RUN THIS FIRST!!

-- Quick check that all Bronze tables still exist
-- Run this at the start of each session to confirm state
-- If any table shows zero it needs to be recreated

In [0]:
%sql
SELECT 'bronze_manta_ray_raw' AS table_name, count(*) AS rows FROM mcphersonsharyn_bronze.bronze_manta_ray_raw
UNION ALL
SELECT 'bronze_whale_shark_raw', count(*) FROM mcphersonsharyn_bronze.bronze_whale_shark_raw
UNION ALL
SELECT 'bronze_sperm_whale_raw', count(*) FROM mcphersonsharyn_bronze.bronze_sperm_whale_raw
UNION ALL
SELECT 'bronze_green_turtle_raw', count(*) FROM mcphersonsharyn_bronze.bronze_green_turtle_raw
ORDER BY table_name;

-- Validates all four species tracking tables still exist



In [0]:
%python
# Bronze SST master — recreated with source filename
# input_file_name() captures filename for year and month extraction in Silver
# This replaces the SQL CREATE TABLE which could not capture _metadata

from pyspark.sql.functions import input_file_name

df = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load("/Volumes/workspace/raw/environmental_data/SST*/") \
  .withColumn("source_file", input_file_name())

df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_sst_master")

print(f"Rows written: {df.count()}")
print("Sample filenames:")
df.select("source_file").distinct().show(5, truncate=False)
# input_file_name() gives us the full file path
# From this we extract SST-Jan1993 pattern to get year and month
# overwrite mode recreates the table with correct schema

In [0]:
%python
# Bronze SST master — rebuilt with year extracted from folder path
# Unity Catalog does not support input_file_name()
# Instead we read each year folder separately and add year manually

from pyspark.sql.functions import lit

# Read all years and tag each with year from folder name
years = list(range(1993, 2027))
dfs = []

for year in years:
    try:
        df_year = spark.read.format("csv") \
          .option("header", "true") \
          .option("inferSchema", "true") \
          .load(f"/Volumes/workspace/raw/environmental_data/SST*{year}*/") \
          .withColumn("year", lit(year))
        dfs.append(df_year)
        print(f"Year {year} loaded")
    except Exception as e:
        print(f"Year {year} skipped: {e}")

# Combine all years
from functools import reduce
from pyspark.sql import DataFrame
df_all = reduce(DataFrame.union, dfs)

df_all.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_sst_master")

print(f"Total rows written: {df_all.count()}")
# Reads each SST year folder separately
# Adds year column directly — no filename parsing needed
# Month will be added in Silver using the filename pattern

In [0]:
%python
from pyspark.sql.functions import lit
from functools import reduce
from pyspark.sql import DataFrame

# Define schema explicitly to avoid inference issues
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType, StringType

schema = StructType([
    StructField("Latitude", DoubleType(), True),
    StructField(" Longitude", DoubleType(), True),
    StructField(" Data Value", DoubleType(), True)
])

years = list(range(1993, 2027))
dfs = []

for year in years:
    # Try both naming conventions
    paths = [
        f"/Volumes/workspace/raw/environmental_data/SST-{year}/",
        f"/Volumes/workspace/raw/environmental_data/SST - {year}/",
        f"/Volumes/workspace/raw/environmental_data/SST- {year}/"
    ]
    for path in paths:
        try:
            df_year = spark.read.format("csv") \
              .option("header", "true") \
              .schema(schema) \
              .load(path) \
              .withColumn("year", lit(year)) \
              .withColumnRenamed("Latitude", "latitude") \
              .withColumnRenamed(" Longitude", "longitude") \
              .withColumnRenamed(" Data Value", "sst_raw")
            count = df_year.count()
            if count > 0:
                dfs.append(df_year)
                print(f"Year {year} loaded from {path} — {count} rows")
                break
        except Exception as e:
            continue

df_all = reduce(DataFrame.union, dfs)

df_all.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_sst_master")

print(f"Total rows: {df_all.count()}")
# Uses explicit schema to avoid inference errors
# Tries all three SST folder naming conventions
# Adds year column directly — used in Silver for time series analysis

In [0]:
%python
# Drop existing table first to clear old schema
spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_bronze.bronze_sst_master")
print("Table dropped")

In [0]:
%python
from pyspark.sql.functions import lit
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, DoubleType

schema = StructType([
    StructField("Latitude", DoubleType(), True),
    StructField(" Longitude", DoubleType(), True),
    StructField(" Data Value", DoubleType(), True)
])

years = list(range(1993, 2027))
dfs = []

for year in years:
    paths = [
        f"/Volumes/workspace/raw/environmental_data/SST-{year}/",
        f"/Volumes/workspace/raw/environmental_data/SST - {year}/",
        f"/Volumes/workspace/raw/environmental_data/SST- {year}/"
    ]
    for path in paths:
        try:
            df_year = spark.read.format("csv") \
              .option("header", "true") \
              .schema(schema) \
              .load(path) \
              .withColumn("year", lit(year)) \
              .withColumnRenamed("Latitude", "latitude") \
              .withColumnRenamed(" Longitude", "longitude") \
              .withColumnRenamed(" Data Value", "sst_raw")
            count = df_year.count()
            if count > 0:
                dfs.append(df_year)
                print(f"Year {year} loaded — {count} rows")
                break
        except Exception as e:
            continue

df_all = reduce(DataFrame.union, dfs)

df_all.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_sst_master")

print(f"Total rows: {df_all.count()}")
# Drops existing table first to clear conflicting schema
# Recreates with year column added directly
# Month added in Silver — extracted from file content patterns

In [0]:
%python
# List all SST folders to see exact naming
import subprocess
result = spark.sql("LIST '/Volumes/workspace/raw/environmental_data/'")
sst_folders = result.filter(result.name.startswith("SST"))
sst_folders.show(50, truncate=False)
# Shows exact folder names for all SST years
# We need these to build correct paths


In [0]:
%python
from pyspark.sql.functions import lit
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, DoubleType

schema = StructType([
    StructField("Latitude", DoubleType(), True),
    StructField(" Longitude", DoubleType(), True),
    StructField(" Data Value", DoubleType(), True)
])

# Exact folder paths from the volume listing
sst_folders = {
    1993: "SST - 1993", 1994: "SST-1994", 1995: "SST-1995",
    1996: "SST-1996", 1997: "SST-1997", 1998: "SST-1998",
    1999: "SST-1999", 2000: "SST-2000", 2001: "SST-2001",
    2002: "SST-2002", 2003: "SST-2003", 2004: "SST-2004",
    2005: "SST-2005", 2006: "SST-2006", 2007: "SST-2007",
    2008: "SST-2008", 2009: "SST- 2009", 2010: "SST - 2010",
    2011: "SST-2011", 2012: "SST - 2012", 2013: "SST - 2013",
    2014: "SST - 2014", 2015: "SST - 2015", 2016: "SST-2016",
    2017: "SST-2017", 2018: "SST-2018", 2019: "SST-2019",
    2020: "SST-2020", 2021: "SST - 2021", 2022: "SST - 2022",
    2023: "SST - 2023", 2024: "SST - 2024", 2025: "SST - 2025",
    2026: "SST-2026"
}

base = "/Volumes/workspace/raw/environmental_data/"
dfs = []

for year, folder in sst_folders.items():
    try:
        path = f"{base}{folder}/"
        df_year = spark.read.format("csv") \
          .option("header", "true") \
          .schema(schema) \
          .load(path) \
          .withColumn("year", lit(year)) \
          .withColumnRenamed("Latitude", "latitude") \
          .withColumnRenamed(" Longitude", "longitude") \
          .withColumnRenamed(" Data Value", "sst_raw")
        dfs.append(df_year)
        print(f"Year {year} loaded from {folder}")
    except Exception as e:
        print(f"Year {year} failed: {e}")

df_all = reduce(DataFrame.union, dfs)

spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_bronze.bronze_sst_master")

df_all.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_sst_master")

print(f"Total rows written: {df_all.count()}")
# Uses exact folder names from the volume listing
# Handles all three naming conventions explicitly
# year column added directly — no filename parsing needed

In [0]:
%python
from pyspark.sql.functions import lit
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, DoubleType

schema = StructType([
    StructField("Latitude", DoubleType(), True),
    StructField(" Longitude", DoubleType(), True),
    StructField(" Data Value", DoubleType(), True)
])

# CC folders named as plain year numbers 2002 to 2026
cc_years = list(range(2002, 2027))
base = "/Volumes/workspace/raw/environmental_data/"
dfs = []

for year in cc_years:
    try:
        path = f"{base}{year}/"
        df_year = spark.read.format("csv") \
          .option("header", "true") \
          .schema(schema) \
          .option("pathGlobFilter", "CC-*.csv") \
          .load(path) \
          .withColumn("year", lit(year)) \
          .withColumnRenamed("Latitude", "latitude") \
          .withColumnRenamed(" Longitude", "longitude") \
          .withColumnRenamed(" Data Value", "cc_raw")
        dfs.append(df_year)
        print(f"Year {year} loaded")
    except Exception as e:
        print(f"Year {year} failed: {e}")

df_all = reduce(DataFrame.union, dfs)

spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_bronze.bronze_cc_master")

df_all.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_cc_master")

print(f"Total rows written: {df_all.count()}")
# CC folders named as plain year numbers 2002 to 2026
# pathGlobFilter excludes FullYear files
# year column added directly for Silver time series analysis
# FullYear files excluded by CC-*.csv pattern

In [0]:
%python
# Check what is inside SST-1994 folder
files = spark.sql("LIST '/Volumes/workspace/raw/environmental_data/SST-1994/'")
files.show(5, truncate=False)
# Shows if files are directly in the folder or in a subfolder


In [0]:
%python
from pyspark.sql.functions import lit
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, DoubleType

schema = StructType([
    StructField("Latitude", DoubleType(), True),
    StructField(" Longitude", DoubleType(), True),
    StructField(" Data Value", DoubleType(), True)
])

sst_folders = {
    1993: "SST - 1993", 1994: "SST-1994", 1995: "SST-1995",
    1996: "SST-1996", 1997: "SST-1997", 1998: "SST-1998",
    1999: "SST-1999", 2000: "SST-2000", 2001: "SST-2001",
    2002: "SST-2002", 2003: "SST-2003", 2004: "SST-2004",
    2005: "SST-2005", 2006: "SST-2006", 2007: "SST-2007",
    2008: "SST-2008", 2009: "SST- 2009", 2010: "SST - 2010",
    2011: "SST-2011", 2012: "SST - 2012", 2013: "SST - 2013",
    2014: "SST - 2014", 2015: "SST - 2015", 2016: "SST-2016",
    2017: "SST-2017", 2018: "SST-2018", 2019: "SST-2019",
    2020: "SST-2020", 2021: "SST - 2021", 2022: "SST - 2022",
    2023: "SST - 2023", 2024: "SST - 2024", 2025: "SST - 2025",
    2026: "SST-2026"
}

base = "/Volumes/workspace/raw/environmental_data/"
dfs = []

for year, folder in sst_folders.items():
    try:
        # Use wildcard to go into nested subfolder automatically
        path = f"{base}{folder}/*/"
        df_year = spark.read.format("csv") \
          .option("header", "true") \
          .option("recursiveFileLookup", "true") \
          .schema(schema) \
          .load(path) \
          .withColumn("year", lit(year)) \
          .withColumnRenamed("Latitude", "latitude") \
          .withColumnRenamed(" Longitude", "longitude") \
          .withColumnRenamed(" Data Value", "sst_raw")
        count = df_year.count()
        if count > 0:
            dfs.append(df_year)
            print(f"Year {year} loaded — {count} rows")
        else:
            print(f"Year {year} — zero rows, skipping")
    except Exception as e:
        print(f"Year {year} failed: {e}")

df_all = reduce(DataFrame.union, dfs)

spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_bronze.bronze_sst_master")

df_all.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_sst_master")

print(f"Total rows written: {df_all.count()}")
# Uses /*/ to navigate into the nested subfolder
# recursiveFileLookup handles any further nesting
# year column added directly from the folder mapping dictionary

In [0]:
%sql
SELECT
  year,
  count(*) AS rows,
  min(latitude) AS min_lat,
  max(latitude) AS max_lat,
  min(sst_raw) AS min_sst,
  max(sst_raw) AS max_sst
FROM mcphersonsharyn_bronze.bronze_sst_master
GROUP BY year
ORDER BY year;
-- Confirms all 33 years present with correct row counts
-- Each year should show approximately 89,424 or 96,876 rows
-- min_sst should not show -999 sentinel values dominating

In [0]:
%sql
SELECT
  year,
  count(*) AS rows
FROM mcphersonsharyn_bronze.bronze_cc_master
GROUP BY year
ORDER BY year;
-- Confirms all 25 CC years present
-- 2002 should show fewer rows — July to December only
-- Each year should show approximately 89,424 or 96,876 rows

In [0]:
%sql
SELECT 'bronze_sst_master' AS table_name, count(*) AS rows FROM mcphersonsharyn_bronze.bronze_sst_master
UNION ALL
SELECT 'bronze_cc_master', count(*) FROM mcphersonsharyn_bronze.bronze_cc_master
UNION ALL
SELECT 'bronze_cmems_raw', count(*) FROM mcphersonsharyn_bronze.bronze_cmems_raw
UNION ALL
SELECT 'bronze_enso_raw', count(*) FROM mcphersonsharyn_bronze.bronze_enso_raw
UNION ALL
SELECT 'bronze_microplastics_raw', count(*) FROM mcphersonsharyn_bronze.bronze_microplastics_raw
UNION ALL
SELECT 'bronze_microplastics_aomi_raw', count(*) FROM mcphersonsharyn_bronze.bronze_microplastics_aomi_raw
UNION ALL
SELECT 'bronze_coral_bleaching_raw', count(*) FROM mcphersonsharyn_bronze.bronze_coral_bleaching_raw
UNION ALL
SELECT 'bronze_fishing_watch_raw', count(*) FROM mcphersonsharyn_bronze.bronze_fishing_watch_raw
ORDER BY table_name;
-- Final Bronze validation before moving to Silver
-- All tables should show non-zero row counts
-- SST should show 3,085,128 and CC should show 2,235,600

In [0]:
%python
from pyspark.sql.functions import lit, input_file_name, regexp_extract
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, DoubleType

schema = StructType([
    StructField("Latitude", DoubleType(), True),
    StructField(" Longitude", DoubleType(), True),
    StructField(" Data Value", DoubleType(), True)
])

month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

sst_folders = {
    1993: "SST - 1993", 1994: "SST-1994", 1995: "SST-1995",
    1996: "SST-1996", 1997: "SST-1997", 1998: "SST-1998",
    1999: "SST-1999", 2000: "SST-2000", 2001: "SST-2001",
    2002: "SST-2002", 2003: "SST-2003", 2004: "SST-2004",
    2005: "SST-2005", 2006: "SST-2006", 2007: "SST-2007",
    2008: "SST-2008", 2009: "SST- 2009", 2010: "SST - 2010",
    2011: "SST-2011", 2012: "SST - 2012", 2013: "SST - 2013",
    2014: "SST - 2014", 2015: "SST - 2015", 2016: "SST-2016",
    2017: "SST-2017", 2018: "SST-2018", 2019: "SST-2019",
    2020: "SST-2020", 2021: "SST - 2021", 2022: "SST - 2022",
    2023: "SST - 2023", 2024: "SST - 2024", 2025: "SST - 2025",
    2026: "SST-2026"
}

base = "/Volumes/workspace/raw/environmental_data/"
dfs = []

for year, folder in sst_folders.items():
    for month_name, month_num in month_map.items():
        try:
            path = f"{base}{folder}/*/SST-{month_name}{year}.csv"
            df_month = spark.read.format("csv") \
              .option("header", "true") \
              .schema(schema) \
              .load(path) \
              .withColumn("year", lit(year)) \
              .withColumn("month", lit(month_num)) \
              .withColumnRenamed("Latitude", "latitude") \
              .withColumnRenamed(" Longitude", "longitude") \
              .withColumnRenamed(" Data Value", "sst_raw")
            count = df_month.count()
            if count > 0:
                dfs.append(df_month)
        except Exception as e:
            pass

    print(f"Year {year} complete")

df_all = reduce(DataFrame.union, dfs)

spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_bronze.bronze_sst_master")

df_all.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_sst_master")

print(f"Total rows written: {df_all.count()}")
# Reads each monthly file individually using exact filename pattern
# SST-Jan1993.csv, SST-Feb1993.csv etc
# Adds both year and month columns directly
# This gives us complete time series in Bronze for Silver to use

In [0]:
%python
# Check what files are actually in one year folder
files = spark.sql("LIST '/Volumes/workspace/raw/environmental_data/SST - 1993/SST-1993/'")
files.show(20, truncate=False)
# Shows exact filenames to confirm naming pattern
# Are they SST-Jan1993.csv or a different format?

In [0]:
%python
from pyspark.sql.functions import lit
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, DoubleType

schema = StructType([
    StructField("Latitude", DoubleType(), True),
    StructField(" Longitude", DoubleType(), True),
    StructField(" Data Value", DoubleType(), True)
])

month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

sst_folders = {
    1993: "SST - 1993", 1994: "SST-1994", 1995: "SST-1995",
    1996: "SST-1996", 1997: "SST-1997", 1998: "SST-1998",
    1999: "SST-1999", 2000: "SST-2000", 2001: "SST-2001",
    2002: "SST-2002", 2003: "SST-2003", 2004: "SST-2004",
    2005: "SST-2005", 2006: "SST-2006", 2007: "SST-2007",
    2008: "SST-2008", 2009: "SST- 2009", 2010: "SST - 2010",
    2011: "SST-2011", 2012: "SST - 2012", 2013: "SST - 2013",
    2014: "SST - 2014", 2015: "SST - 2015", 2016: "SST-2016",
    2017: "SST-2017", 2018: "SST-2018", 2019: "SST-2019",
    2020: "SST-2020", 2021: "SST - 2021", 2022: "SST - 2022",
    2023: "SST - 2023", 2024: "SST - 2024", 2025: "SST - 2025",
    2026: "SST-2026"
}

base = "/Volumes/workspace/raw/environmental_data/"
dfs = []

for year, folder in sst_folders.items():
    year_rows = 0
    for month_name, month_num in month_map.items():
        filename = f"SST-{month_name}{year}.csv"
        # Try both path levels — direct and nested subfolder
        paths = [
            f"{base}{folder}/{filename}",
            f"{base}{folder}/{folder.strip()}/{filename}",
            f"{base}{folder}/SST-{year}/{filename}",
            f"{base}{folder}/SST-{year.strip()}/{filename}"
        ]
        loaded = False
        for path in paths:
            try:
                df_month = spark.read.format("csv") \
                  .option("header", "true") \
                  .schema(schema) \
                  .load(path) \
                  .withColumn("year", lit(year)) \
                  .withColumn("month", lit(month_num)) \
                  .withColumnRenamed("Latitude", "latitude") \
                  .withColumnRenamed(" Longitude", "longitude") \
                  .withColumnRenamed(" Data Value", "sst_raw")
                count = df_month.count()
                if count > 0:
                    dfs.append(df_month)
                    year_rows += count
                    loaded = True
                    break
            except Exception as e:
                continue
        if not loaded:
            print(f"  Missing: {year} {month_name}")
    print(f"Year {year} complete — {year_rows} rows")

df_all = reduce(DataFrame.union, dfs)

spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_bronze.bronze_sst_master")

df_all.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_sst_master")

print(f"Total rows written: {df_all.count()}")
# Tries multiple path patterns for each monthly file
# Handles both direct and nested subfolder structures
# Prints missing files so we can investigate any gaps

In [0]:
%python
from pyspark.sql.functions import lit
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, DoubleType

schema = StructType([
    StructField("Latitude", DoubleType(), True),
    StructField(" Longitude", DoubleType(), True),
    StructField(" Data Value", DoubleType(), True)
])

month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

sst_folders = {
    1993: "SST - 1993", 1994: "SST-1994", 1995: "SST-1995",
    1996: "SST-1996", 1997: "SST-1997", 1998: "SST-1998",
    1999: "SST-1999", 2000: "SST-2000", 2001: "SST-2001",
    2002: "SST-2002", 2003: "SST-2003", 2004: "SST-2004",
    2005: "SST-2005", 2006: "SST-2006", 2007: "SST-2007",
    2008: "SST-2008", 2009: "SST- 2009", 2010: "SST - 2010",
    2011: "SST-2011", 2012: "SST - 2012", 2013: "SST - 2013",
    2014: "SST - 2014", 2015: "SST - 2015", 2016: "SST-2016",
    2017: "SST-2017", 2018: "SST-2018", 2019: "SST-2019",
    2020: "SST-2020", 2021: "SST - 2021", 2022: "SST - 2022",
    2023: "SST - 2023", 2024: "SST - 2024", 2025: "SST - 2025",
    2026: "SST-2026"
}

base = "/Volumes/workspace/raw/environmental_data/"
dfs = []

for year, folder in sst_folders.items():
    year_rows = 0
    for month_name, month_num in month_map.items():
        filename = f"SST-{month_name}{year}.csv"
        # Try multiple path patterns — direct and nested subfolder
        paths = [
            f"{base}{folder}/{filename}",
            f"{base}{folder}/SST-{year}/{filename}",
            f"{base}{folder}/SST {year}/{filename}",
            f"{base}{folder}/SST- {year}/{filename}"
        ]
        loaded = False
        for path in paths:
            try:
                df_month = spark.read.format("csv") \
                  .option("header", "true") \
                  .schema(schema) \
                  .load(path) \
                  .withColumn("year", lit(year)) \
                  .withColumn("month", lit(month_num)) \
                  .withColumnRenamed("Latitude", "latitude") \
                  .withColumnRenamed(" Longitude", "longitude") \
                  .withColumnRenamed(" Data Value", "sst_raw")
                count = df_month.count()
                if count > 0:
                    dfs.append(df_month)
                    year_rows += count
                    loaded = True
                    break
            except Exception as e:
                continue
        if not loaded:
            print(f"  Missing: {year} {month_name}")
    print(f"Year {year} complete — {year_rows} rows")

df_all = reduce(DataFrame.union, dfs)

spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_bronze.bronze_sst_master")

df_all.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_sst_master")

print(f"Total rows written: {df_all.count()}")
# Fixed: year is integer so no .strip() needed
# Tries four path patterns for each monthly file
# Prints any missing month/year combinations

In [0]:
%python
# Check exact subfolder name inside SST- 2009 and SST - 2014
files_2009 = spark.sql("LIST '/Volumes/workspace/raw/environmental_data/SST- 2009/'")
files_2009.show(5, truncate=False)

files_2014 = spark.sql("LIST '/Volumes/workspace/raw/environmental_data/SST - 2014/'")
files_2014.show(5, truncate=False)
# Shows exact subfolder names so we can add correct paths

In [0]:
%python
# Check all folders that had missing months to find their filename patterns
problem_years = [2009, 2011, 2013, 2014, 2015, 2016, 2020, 2022, 2025]
folders = {
    2009: "SST- 2009", 2011: "SST-2011", 2013: "SST - 2013",
    2014: "SST - 2014", 2015: "SST - 2015", 2016: "SST-2016",
    2020: "SST-2020", 2022: "SST - 2022", 2025: "SST - 2025"
}

base = "/Volumes/workspace/raw/environmental_data/"
for year, folder in folders.items():
    files = spark.sql(f"LIST '{base}{folder}/'")
    print(f"\n=== {year} ({folder}) ===")
    files.show(3, truncate=False)
# Shows filename patterns for all problem years
# Some use 2-digit years, some have spaces, some have subfolders

In [0]:
%python
from pyspark.sql.functions import lit, col, regexp_extract, when
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType

schema = StructType([
    StructField("Latitude", DoubleType(), True),
    StructField(" Longitude", DoubleType(), True),
    StructField(" Data Value", DoubleType(), True)
])

sst_folders = {
    1993: "SST - 1993", 1994: "SST-1994", 1995: "SST-1995",
    1996: "SST-1996", 1997: "SST-1997", 1998: "SST-1998",
    1999: "SST-1999", 2000: "SST-2000", 2001: "SST-2001",
    2002: "SST-2002", 2003: "SST-2003", 2004: "SST-2004",
    2005: "SST-2005", 2006: "SST-2006", 2007: "SST-2007",
    2008: "SST-2008", 2009: "SST- 2009", 2010: "SST - 2010",
    2011: "SST-2011", 2012: "SST - 2012", 2013: "SST - 2013",
    2014: "SST - 2014", 2015: "SST - 2015", 2016: "SST-2016",
    2017: "SST-2017", 2018: "SST-2018", 2019: "SST-2019",
    2020: "SST-2020", 2021: "SST - 2021", 2022: "SST - 2022",
    2023: "SST - 2023", 2024: "SST - 2024", 2025: "SST - 2025",
    2026: "SST-2026"
}

base = "/Volumes/workspace/raw/environmental_data/"
dfs = []

for year, folder in sst_folders.items():
    try:
        # Read all CSV files in folder recursively
        # Then extract month from the file listing separately
        all_files = spark.sql(f"LIST '{base}{folder}/'")
        csv_files = [
            row.path for row in all_files.collect()
            if row.name.endswith('.csv')
            and 'FullYear' not in row.name
            and 'fullyear' not in row.name.lower()
        ]

        # Also check one level deeper
        try:
            subfolder_files = spark.sql(
                f"LIST '{base}{folder}/SST-{year}/'"
            )
            csv_files += [
                row.path for row in subfolder_files.collect()
                if row.name.endswith('.csv')
                and 'FullYear' not in row.name
                and 'fullyear' not in row.name.lower()
            ]
        except:
            pass

        year_rows = 0
        for filepath in csv_files:
            # Extract month name from filename using regex
            import re
            match = re.search(
                r'SST\s*-?\s*(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)',
                filepath, re.IGNORECASE
            )
            if not match:
                print(f"  Could not extract month from: {filepath}")
                continue

            month_name = match.group(1).capitalize()
            month_map = {
                'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
                'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
                'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
            }
            month_num = month_map.get(month_name, 0)
            if month_num == 0:
                continue

            try:
                df_month = spark.read.format("csv") \
                  .option("header", "true") \
                  .schema(schema) \
                  .load(filepath) \
                  .withColumn("year", lit(year)) \
                  .withColumn("month", lit(month_num)) \
                  .withColumnRenamed("Latitude", "latitude") \
                  .withColumnRenamed(" Longitude", "longitude") \
                  .withColumnRenamed(" Data Value", "sst_raw")
                count = df_month.count()
                if count > 0:
                    dfs.append(df_month)
                    year_rows += count
            except Exception as e:
                print(f"  Failed to read: {filepath} — {e}")

        print(f"Year {year} complete — {year_rows} rows")

    except Exception as e:
        print(f"Year {year} failed: {e}")

from functools import reduce
from pyspark.sql import DataFrame
df_all = reduce(DataFrame.union, dfs)

spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_bronze.bronze_sst_master")

df_all.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("mcphersonsharyn_bronze.bronze_sst_master")

print(f"Total rows written: {df_all.count()}")
# Lists all CSV files in each folder dynamically
# Extracts month name from actual filename using regex
# Handles all naming variations — 2-digit years, spaces, subfolders
# FullYear files excluded before processing

In [0]:
%python
# Fix: update regex to handle underscore separator in some filenames
import re

# Test the fix
test_files = [
    'SST_Jun09.csv', 'SST_Sep09.csv', 'SST_Mar2013.csv',
    'SST_Feb2015.csv', 'SST_May2015.csv', 'SST_May2022.csv'
]

for f in test_files:
    match = re.search(
        r'SST[\s\-_]+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)',
        f, re.IGNORECASE
    )
    if match:
        print(f"{f} → {match.group(1)}")
    else:
        print(f"{f} → NO MATCH")
# Tests updated regex against all problem filenames
# [\s\-_]+ matches space, dash or underscore between SST and month name

In [0]:
%python
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, DoubleType
from functools import reduce
from pyspark.sql import DataFrame
import re

schema = StructType([
    StructField("Latitude", DoubleType(), True),
    StructField(" Longitude", DoubleType(), True),
    StructField(" Data Value", DoubleType(), True)
])

month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

sst_folders = {
    1993: "SST - 1993", 1994: "SST-1994", 1995: "SST-1995",
    1996: "SST-1996", 1997: "SST-1997", 1998: "SST-1998",
    1999: "SST-1999", 2000: "SST-2000", 2001: "SST-2001",
    2002: "SST-2002", 2003: "SST-2003", 2004: "SST-2004",
    2005: "SST-2005", 2006: "SST-2006", 2007: "SST-2007",
    2008: "SST-2008", 2009: "SST- 2009", 2010: "SST - 2010",
    2011: "SST-2011", 2012: "SST - 2012", 2013: "SST - 2013",
    2014: "SST - 2014", 2015: "SST - 2015", 2016: "SST-2016",
    2017: "SST-2017", 2018: "SST-2018", 2019: "SST-2019",
    2020: "SST-2020", 2021: "SST - 2021", 2022: "SST - 2022",
    2023: "SST - 2023", 2024: "SST - 2024", 2025: "SST - 2025",
    2026: "SST-2026"
}

base = "/Volumes/workspace/raw/environmental_data/"
dfs = []

def get_csv_files(folder_path):
    result = []
    try:
        files = spark.sql("LIST '" + folder_path + "'")
        for row in files.collect():
            if row.name.endswith('.csv') and 'fullyear' not in row.name.lower():
                result.append(row.path)
    except Exception as e:
        pass
    return result

def get_month_num(filepath):
    pattern = r'SST[\s\-_]+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
    match = re.search(pattern, filepath, re.IGNORECASE)
    if match:
        return month_map.get(match.group(1).capitalize(), 0)
    return 0

for year, folder in sst_folders.items():
    year_rows = 0
    folder_path = base + folder + "/"
    subfolder_path = base + folder + "/SST-" + str(year) + "/"

    csv_files = get_csv_files(folder_path)
    csv_files += get_csv_files(subfolder_path)

    for filepath in csv_files:
        month_num = get_month_num(filepath)
        if month_num == 0:
            print("Cannot extract month: " + filepath)
            continue
        try:
            df_month = spark.read.format("csv") \
                .option("header", "true") \
                .schema(schema) \
                .load(filepath) \
                .withColumn("year", lit(year)) \
                .withColumn("month", lit(month_num)) \
                .withColumnRenamed("Latitude", "latitude") \
                .withColumnRenamed(" Longitude", "longitude") \
                .withColumnRenamed(" Data Value", "sst_raw")
            count = df_month.count()
            if count > 0:
                dfs.append(df_month)
                year_rows += count
        except Exception as e:
            print("Failed: " + filepath)

    print("Year " + str(year) + " complete — " + str(year_rows) + " rows")

df_all = reduce(DataFrame.union, dfs)
spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_bronze.bronze_sst_master")
df_all.write.format("delta").mode("overwrite").saveAsTable("mcphersonsharyn_bronze.bronze_sst_master")
print("Total rows written: " + str(df_all.count()))
# Refactored into helper functions to avoid syntax issues
# get_csv_files lists all valid CSV files in a folder
# get_month_num extracts month number from any filename pattern
# Handles dash, underscore and space separators

In [0]:
%sql
SELECT
  year,
  count(DISTINCT month) AS months,
  count(*) AS rows
FROM mcphersonsharyn_bronze.bronze_sst_master
GROUP BY year
ORDER BY year;
-- Confirms how many months are present per year
-- All years except 2026 should show 12 months
-- 2026 should show 8 months (Jan to Aug)

In [0]:
%python
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, DoubleType
from functools import reduce
from pyspark.sql import DataFrame
import re

schema = StructType([
    StructField("Latitude", DoubleType(), True),
    StructField(" Longitude", DoubleType(), True),
    StructField(" Data Value", DoubleType(), True)
])

month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}

base = "/Volumes/workspace/raw/environmental_data/"
cc_years = list(range(2002, 2027))
dfs = []

def get_csv_files(folder_path):
    result = []
    try:
        files = spark.sql("LIST '" + folder_path + "'")
        for row in files.collect():
            if row.name.endswith('.csv') and 'fullyear' not in row.name.lower() and 'FullYear' not in row.name:
                result.append(row.path)
    except Exception as e:
        pass
    return result

def get_month_num(filepath):
    pattern = r'CC[\s\-_]+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
    match = re.search(pattern, filepath, re.IGNORECASE)
    if match:
        return month_map.get(match.group(1).capitalize(), 0)
    return 0

for year in cc_years:
    year_rows = 0
    folder_path = base + str(year) + "/"
    csv_files = get_csv_files(folder_path)

    for filepath in csv_files:
        month_num = get_month_num(filepath)
        if month_num == 0:
            print("Cannot extract month: " + filepath)
            continue
        try:
            df_month = spark.read.format("csv") \
                .option("header", "true") \
                .schema(schema) \
                .load(filepath) \
                .withColumn("year", lit(year)) \
                .withColumn("month", lit(month_num)) \
                .withColumnRenamed("Latitude", "latitude") \
                .withColumnRenamed(" Longitude", "longitude") \
                .withColumnRenamed(" Data Value", "cc_raw")
            count = df_month.count()
            if count > 0:
                dfs.append(df_month)
                year_rows += count
        except Exception as e:
            print("Failed: " + filepath)

    print("Year " + str(year) + " complete — " + str(year_rows) + " rows")

df_all = reduce(DataFrame.union, dfs)
spark.sql("DROP TABLE IF EXISTS mcphersonsharyn_bronze.bronze_cc_master")
df_all.write.format("delta").mode("overwrite").saveAsTable("mcphersonsharyn_bronze.bronze_cc_master")
print("Total rows written: " + str(df_all.count()))
# Same approach as SST — reads actual files and extracts month from filename
# CC folders named as plain year numbers 2002 to 2026
# FullYear files excluded
# Expected: approximately 2,235,600 rows

In [0]:
%sql
SELECT
  year,
  count(DISTINCT month) AS months,
  count(*) AS rows
FROM mcphersonsharyn_bronze.bronze_cc_master
GROUP BY year
ORDER BY year;
-- Confirms month coverage per year
-- 2002 should show 6 months, 2024 should show 11
-- All other years should show 12

In [0]:
%sql
SELECT 'bronze_sst_master' AS table_name, count(*) AS rows FROM mcphersonsharyn_bronze.bronze_sst_master
UNION ALL
SELECT 'bronze_cc_master', count(*) FROM mcphersonsharyn_bronze.bronze_cc_master
UNION ALL
SELECT 'bronze_cmems_raw', count(*) FROM mcphersonsharyn_bronze.bronze_cmems_raw
UNION ALL
SELECT 'bronze_enso_raw', count(*) FROM mcphersonsharyn_bronze.bronze_enso_raw
UNION ALL
SELECT 'bronze_microplastics_raw', count(*) FROM mcphersonsharyn_bronze.bronze_microplastics_raw
UNION ALL
SELECT 'bronze_microplastics_aomi_raw', count(*) FROM mcphersonsharyn_bronze.bronze_microplastics_aomi_raw
UNION ALL
SELECT 'bronze_coral_bleaching_raw', count(*) FROM mcphersonsharyn_bronze.bronze_coral_bleaching_raw
UNION ALL
SELECT 'bronze_fishing_watch_raw', count(*) FROM mcphersonsharyn_bronze.bronze_fishing_watch_raw
ORDER BY table_name;
-- Final Bronze validation before Silver
-- All tables confirmed with correct row counts

In [0]:
%sql
INSERT INTO mcphersonsharyn_bronze.bronze_sst_master
SELECT
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  CAST(sst AS DOUBLE) AS sst_raw,
  YEAR(TO_TIMESTAMP(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS year,
  MONTH(TO_TIMESTAMP(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS month
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/SST-GulfOfMexico/SST-gulfofmexico.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)
WHERE time != 'UTC'
AND latitude != 'degrees_north'
AND sst != 'NaN';
-- Inserts Gulf of Mexico SST into existing bronze_sst_master
-- Extracts year and month from ISO timestamp
-- Excludes units row and NaN values at Bronze stage
-- zlev column excluded — surface only confirmed
-- Expected: approximately 1.5 million rows

In [0]:
%sql
SELECT
  count(*) AS total_rows,
  min(longitude) AS min_lon,
  max(longitude) AS max_lon,
  count(DISTINCT year) AS distinct_years,
  min(year) AS earliest_year,
  max(year) AS latest_year
FROM mcphersonsharyn_bronze.bronze_sst_master;
-- Confirms Gulf of Mexico rows added
-- Total should be 3,010,608 + 1,642,752 = 4,653,360
-- Longitude should now include Gulf of Mexico values around -97 to -72

In [0]:
%sql
SELECT DISTINCT longitude
FROM mcphersonsharyn_bronze.bronze_sst_master
WHERE longitude BETWEEN -97 AND -72
AND year = 2009
AND month = 6
ORDER BY longitude
LIMIT 20;
-- Confirms Gulf of Mexico longitude values now exist in Bronze
-- Should show 0.25 degree resolution values between -97 and -72

In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/';
-- Find the exact folder name for Indian Ocean SST

In [0]:
%sql
LIST '/Volumes/workspace/raw/environmental_data/SST-IndianOcean/';
-- Confirms Indian Ocean SST file name and size

In [0]:
%sql
INSERT INTO mcphersonsharyn_bronze.bronze_sst_master
SELECT
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  CAST(sst AS DOUBLE) AS sst_raw,
  YEAR(TO_TIMESTAMP(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS year,
  MONTH(TO_TIMESTAMP(time, "yyyy-MM-dd'T'HH:mm:ss'Z'")) AS month
FROM read_files(
  '/Volumes/workspace/raw/environmental_data/SST-IndianOcean/SST-indianocean.csv',
  format => 'csv',
  header => true,
  inferSchema => false
)
WHERE time != 'UTC'
AND latitude != 'degrees_north'
AND sst != 'NaN';
-- Inserts Indian Ocean SST into bronze_sst_master
-- Covers longitude -72 to 125 — green turtle and manta ray habitat
-- Extracts year and month from ISO timestamp
-- NaN values excluded at Bronze stage



In [0]:
%sql
SELECT
  count(*) AS total_rows,
  count(DISTINCT year) AS distinct_years,
  min(longitude) AS min_lon,
  max(longitude) AS max_lon,
  count(DISTINCT ROUND(longitude, 0)) AS distinct_integer_lons
FROM mcphersonsharyn_bronze.bronze_sst_master;
-- Confirms Indian Ocean rows added
-- Total should be 4,653,360 + 33,264,984 = 37,918,344
-- Longitude should now cover -72 to 125